# Mammogram lesion localization and benign/malignant classification
## Imaging-only training notebook — configured for your Google Drive manifest

**Dataset ZIP (Drive):** `/content/drive/MyDrive/mammo_semcom_data/CBIS-DDSM.zip`  → extracted each session to `/content/cbis_ddsm` (local Colab disk)  
**Manifest:** `/content/drive/MyDrive/mammo_semcom_work/cache/manifest.csv`  
**Results:** `/content/drive/MyDrive/mammo_semcom_work/runs/imaging_only_v1`

Training is enabled. Use a Colab GPU and run cells in order.

**What changed versus the first version (paths and manifest handling only; models, losses and metrics are unchanged):**
- The dataset ZIP is extracted to `/content/cbis_ddsm`, the same folder your data-preparation notebook used, so the absolute paths stored in `manifest.csv` resolve again after a runtime reset.
- New pre-flight cell reports which manifest files exist before anything runs.
- Rows with no full-image or ROI file are dropped and listed in `dropped_manifest_rows.csv`, not fatal.
- Unresolvable paths are dropped and listed (fatal only if more than 5% are lost).
- If a ROI file does not match the full-image size but the crop file does (a known CBIS-DDSM ROI/crop mix-up), the two are swapped and logged.
- If patients cross the official train/test files (a known CBIS-DDSM property), the notebook automatically falls back to a patient-level 70/15/15 split and records it.
- Cache fingerprints no longer include file modification times, otherwise re-extracting the ZIP would invalidate the cache and checkpoints every session.

This notebook contains ONLY imaging tasks: registered preprocessing, patient-disjoint splitting, localization, image-level classification, validation, metrics, plots and checkpoints. It contains no transmission, packet simulation, delivery masks, reconstruction, block priority or receiver-utility model.

## Execution and research status
- Python syntax of every cell was checked, and the new path/manifest logic was tested on a small synthetic dataset. PyTorch execution, training and the real manifest have NOT been run by the author of this notebook.
- Architectures are custom research baselines, not verified novel architectures.
- Localization uses the earlier instance-balanced multiscale quadratic objective. Classification uses class-balanced Brier risk. Neither is claimed novel.
- The later proposed decision-regret loss is NOT implemented because it requires a different structured hypothesis model.
- No clinical performance is fabricated. No pretrained weights are used.
- Your schema lacks clinical phenotype descriptors. Assessment/subtlety are NOT used as input features, diagnosis targets, or invented descriptor labels.

## Data units
The manifest is abnormality-level. Multiple records sharing a resolved `full_file` are combined into one image. `roi_file` must be a registered full-image lesion mask. `crop_file` is never used as a mask or independent training image. Abnormality IDs define lesion instances; disconnected calcifications belonging to one record remain one annotated lesion.

Mixed benign/malignant images retain localization supervision but have a missing image diagnosis and are excluded from binary classification loss and metrics.

## References
CBIS-DDSM: Lee et al., Scientific Data 2017, doi:10.1038/sdata.2017.177.  
GMIC global-local mammography: Shen et al., Medical Image Analysis 2021, doi:10.1016/j.media.2020.101908.  
Instance-balancing prior: Kofler et al., blob loss, https://arxiv.org/abs/2205.08209.

In [ ]:
# 1. Environment and Drive mount
import sys, subprocess, importlib.util
if importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
packages={'numpy':'numpy','pandas':'pandas','PIL':'pillow','scipy':'scipy','sklearn':'scikit-learn','matplotlib':'matplotlib','pydicom':'pydicom','torch':'torch'}
missing=[package for module,package in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict
import json, hashlib, random, math, time, warnings, platform
import numpy as np
import pandas as pd
from PIL import Image
import pydicom
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (roc_auc_score,average_precision_score,confusion_matrix,
                             balanced_accuracy_score,f1_score,roc_curve,precision_recall_curve)
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader

@dataclass
class Config:
    data_dir: str='/content/cbis_ddsm'   # local extraction folder (same as the data-preparation notebook)
    zip_path: str='/content/drive/MyDrive/mammo_semcom_data/CBIS-DDSM.zip'
    work_dir: str='/content/drive/MyDrive/mammo_semcom_work'
    manifest: str='/content/drive/MyDrive/mammo_semcom_work/cache/manifest.csv'
    height: int=768
    width: int=512
    batch_size: int=2
    workers: int=0
    seed: int=17
    validation_fraction_of_train: float=.17
    localization_epochs: int=40
    classification_epochs: int=40
    patience: int=8
    learning_rate: float=.01
    weight_decay: float=1e-4
    optimizer: str='sgd'
    resume: bool=True
    rebuild_cache: bool=False
    repartition_all_patients: bool=False   # set True to force a patient-level 70/15/15 split; it is also switched on automatically if official splits share patients
    localization_scales: tuple=(0,1,3)
    balance_instances: bool=True
    mask_threshold_fraction: float=.5
    dicom_percentiles: tuple=(.5,99.5)
    matching_iou: float=.1
    small_lesion_resized_area: int=100

cfg=Config()
RUN_TRAINING=True
RUN_TEST=False   # Enable only after model/tuning/threshold choices are frozen.
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK=Path(cfg.work_dir); RUN=WORK/'runs'/'imaging_only_v1'; CACHE=WORK/'cache'/'imaging_only_v1'
RUN.mkdir(parents=True,exist_ok=True); CACHE.mkdir(parents=True,exist_ok=True)
assert cfg.height%4==0 and cfg.width%4==0

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
seed_all(cfg.seed)
json.dump(asdict(cfg),open(RUN/'config.json','w'),indent=2)
json.dump(dict(python=platform.python_version(),torch=torch.__version__,numpy=np.__version__,device=str(DEVICE)),open(RUN/'environment.json','w'),indent=2)
print('Device:',DEVICE)
if not torch.cuda.is_available(): warnings.warn('CPU fallback is active; full training will be slow. Select a Colab GPU.')
if not Path(cfg.manifest).is_file(): raise FileNotFoundError(cfg.manifest)

In [ ]:
# 1b. Extract the dataset ZIP to local Colab disk (skipped if already extracted this session)
import zipfile, shutil
zip_path=Path(cfg.zip_path); data_dir=Path(cfg.data_dir)
def already_extracted(folder):
    return folder.exists() and next((p for p in folder.rglob('*') if p.is_file()),None) is not None
if already_extracted(data_dir):
    print('Dataset already extracted at',data_dir)
else:
    if not zip_path.is_file(): raise FileNotFoundError(f'ZIP not found: {zip_path}. Check the path in Drive (left sidebar > Files).')
    with zipfile.ZipFile(zip_path) as z:
        needed=sum(i.file_size for i in z.infolist()); members=len(z.infolist())
    free=shutil.disk_usage('/content').free
    print(f'ZIP: {zip_path.stat().st_size/1e9:.2f} GB compressed, {needed/1e9:.2f} GB extracted, {members} entries; free local disk {free/1e9:.1f} GB')
    if needed>0.9*free: raise RuntimeError('Not enough local disk to extract; use a larger runtime or extract a subset.')
    data_dir.mkdir(parents=True,exist_ok=True)
    t0=time.time(); subprocess.check_call(['unzip','-q','-n',str(zip_path),'-d',str(data_dir)]); print(f'Extracted in {time.time()-t0:.0f}s')
print('Extracted files:',sum(1 for p in data_dir.rglob('*') if p.is_file()))


In [ ]:
# 1c. Manifest pre-flight (read-only report; the real validation happens in section 2)
pf=pd.read_csv(cfg.manifest,sep=None,engine='python',dtype={'patient_id':str})
pf.columns=[str(c).strip().lower().replace(' ','_') for c in pf.columns]
print('Manifest:',cfg.manifest); print('Rows:',len(pf)); print('Columns:',pf.columns.tolist())
for column in ['full_file','roi_file','crop_file']:
    if column in pf:
        values=pf[column].dropna().astype(str); exact=values.map(lambda v: Path(v).is_file())
        print(f'{column:10s} empty={pf[column].isna().sum():5d}  exists-as-written={int(exact.sum())}/{len(values)}')
        if len(values): print('   example:',values.iloc[0])
for column in ['lesion_type','split_official','pathology','label']:
    if column in pf: print(column,'->',pf[column].astype(str).value_counts().head(8).to_dict())
if {'patient_id','split_official'}.issubset(pf):
    print('patients appearing in more than one official split:',int((pf.groupby('patient_id').split_official.nunique()>1).sum()))
print('\nNote: "exists-as-written" can be 0 and still be fine if the files are under data_dir with a different prefix; section 2 resolves those by unique path suffix.')


## 2. Manifest mapping and patient partitions
Expected columns: `patient_id, side, view, lesion_type, split_official, abnormality_id, pathology, label, subtlety, assessment, full_file, roi_file, crop_file`.

The code normalizes column names, resolves moved paths, checks label/pathology agreement (0 benign, 1 malignant), and groups full images. Official test patients are preserved. Validation is created from official train patients only when no validation partition exists. Patient conflicts across official splits stop execution. Do not override this check without a documented new split protocol.

The resolver accepts absolute paths, paths relative to data/work/cache, and unambiguous relocated suffixes. It refuses ambiguous duplicates. Exact processed-image duplicates across partitions are checked later; near-duplicate/re-encoding detection remains an additional audit.

**Path handling:** the extracted dataset lives in `/content/cbis_ddsm`. A manifest path is used as written if it exists; otherwise it is matched by the longest unique path suffix inside the extracted folder. Rows that cannot be matched are written to `unresolved_manifest_rows.csv` and dropped (stop if >5%). `crop_file` is used only as a fallback when the listed ROI has the wrong shape, and each swap is logged to `roi_crop_swaps.csv`.

In [ ]:
raw=pd.read_csv(cfg.manifest,sep=None,engine='python',dtype={'patient_id':str})
raw.columns=[str(c).strip().lower().replace(' ','_') for c in raw.columns]
if 'lesion_types' in raw and 'lesion_type' not in raw: raw=raw.rename(columns={'lesion_types':'lesion_type'})
required={'patient_id','full_file','roi_file','lesion_type','split_official','abnormality_id','pathology','label'}
if not required.issubset(raw): raise ValueError(f'Missing {required-set(raw)}; found {raw.columns.tolist()}')
if 'crop_file' not in raw: raw['crop_file']=np.nan
if raw[['patient_id','lesion_type','abnormality_id']].isna().any().any(): raise ValueError('Missing patient/lesion identifiers in manifest')
missing_files=raw[raw.full_file.isna()|raw.roi_file.isna()]
if len(missing_files):
    missing_files.to_csv(RUN/'dropped_manifest_rows.csv',index=False)
    print(f'Dropping {len(missing_files)} manifest rows with no full-image or ROI file (crop-only rows); listed in dropped_manifest_rows.csv')
raw=raw[raw.full_file.notna()&raw.roi_file.notna()].copy()
raw['abnormality_id']=raw.abnormality_id.astype(str)
before=len(raw); raw=raw.drop_duplicates(subset=['patient_id','full_file','roi_file','abnormality_id','lesion_type']).reset_index(drop=True)
if len(raw)<before: print('Removed',before-len(raw),'exact duplicate rows')
raw.patient_id=raw.patient_id.astype(str).str.strip()
ROOTS=[Path(cfg.data_dir),Path(cfg.work_dir),Path(cfg.manifest).parent]; index_by_name=None

def resolve_path(value):
    global index_by_name
    text=str(value).strip().replace(chr(92),'/'); p=Path(text)
    candidates=([p] if p.is_absolute() else [])+[root/text for root in ROOTS]
    for name,root in [('mammo_semcom_data',Path(cfg.data_dir)),('mammo_semcom_work',Path(cfg.work_dir))]:
        if name+'/' in text: candidates.append(root/text.split(name+'/',1)[1])
    for candidate in candidates:
        if candidate.is_file(): return str(candidate.resolve())
    if index_by_name is None:
        print('Indexing file paths once...'); index_by_name=defaultdict(set)
        for root in ROOTS[:1]:   # index only the extracted dataset, not the Drive work folder
            if root.exists():
                for f in root.rglob('*'):
                    if f.is_file(): index_by_name[f.name].add(str(f.resolve()))
    matches=list(index_by_name.get(p.name,set()))
    if len(matches)==1: return matches[0]
    parts=[x for x in text.split('/') if x]
    for n in range(min(len(parts),10),1,-1):
        found=[m for m in matches if m.endswith('/'+'/'.join(parts[-n:]))]
        if len(found)==1: return found[0]
    raise FileNotFoundError(f'Unresolved/ambiguous path {text}; candidates {matches[:3]}')

def safe_resolve(value):
    try: return resolve_path(value)
    except FileNotFoundError: return None
unresolved=[]
for column in ['full_file','roi_file','crop_file']:
    values=raw[column].dropna().astype(str).unique(); mapping={v:safe_resolve(v) for v in values}
    raw[column+'_resolved']=raw[column].astype(str).where(raw[column].notna()).map(mapping)
    if column!='crop_file': unresolved+=[(column,v) for v,r in mapping.items() if r is None]
bad=raw.full_file_resolved.isna()|raw.roi_file_resolved.isna()
print(f'Resolved paths: {int((~bad).sum())}/{len(raw)} abnormality rows')
if bad.any():
    raw[bad].to_csv(RUN/'unresolved_manifest_rows.csv',index=False)
    print(f'{int(bad.sum())} rows have unresolved files (see unresolved_manifest_rows.csv); first examples:',unresolved[:3])
    if bad.mean()>.05: raise FileNotFoundError(f'{bad.mean():.1%} of rows unresolved. Is the ZIP extracted to {cfg.data_dir}, and does it contain the same files as when the manifest was built?')
    raw=raw[~bad].reset_index(drop=True)
# ROI/crop swap lookup: if an ROI file is not the full-image size, the crop file is tried (see read_roi)
ALT_ROI={}
for r in raw.itertuples():
    if isinstance(r.crop_file_resolved,str): ALT_ROI[r.roi_file_resolved]=r.crop_file_resolved

def diagnosis(row):
    aliases={'BENIGN':0.,'BENIGN_WITHOUT_CALLBACK':0.,'MALIGNANT':1.}
    pathology=aliases.get(str(row.pathology).strip().upper().replace(' ','_'),np.nan)
    if pd.isna(row.label): label=np.nan
    else:
        text=str(row.label).strip().upper().replace(' ','_')
        if text in aliases: label=aliases[text]
        else:
            try: label=float(row.label)
            except (TypeError,ValueError): raise ValueError(f'Invalid diagnosis {row.label}')
        if label not in [0.,1.]: raise ValueError(f'Expected 0/1 diagnosis: {row.label}')
    if np.isfinite(label) and np.isfinite(pathology) and label!=pathology: raise ValueError(f'Pathology/label disagree: {row.full_file}')
    return pathology if np.isfinite(pathology) else label

def lesion_type(value):
    value=str(value).lower()
    if 'mass' in value and 'calc' not in value: return 1
    if 'calc' in value and 'mass' not in value: return 2
    raise ValueError(f'Expected one lesion type per abnormality row: {value}')

def official_split(value):
    text=str(value).strip().lower()
    if 'train' in text: return 'train'
    if 'test' in text: return 'test'
    if 'val' in text: return 'val'
    raise ValueError(f'Unknown official split {value}')

raw['diagnosis']=raw.apply(diagnosis,axis=1); raw['type_code']=raw.lesion_type.map(lesion_type)
raw['official']=raw.split_official.map(official_split); rows=[]
for full,group in raw.groupby('full_file_resolved',sort=True):
    if group.patient_id.nunique()!=1: raise ValueError(f'Same image assigned to several patients: {full}')
    labels=set(group.diagnosis.dropna()); mixed=len(labels)>1
    label=next(iter(labels)) if len(labels)==1 and group.diagnosis.notna().all() else np.nan
    rows.append(dict(image_id=hashlib.sha256(full.encode()).hexdigest()[:20],patient_id=group.patient_id.iloc[0],
                     full_file=full,official=group.official.iloc[0],official_conflict=group.official.nunique()!=1,label=label,mixed=mixed,abnormalities=len(group)))
images=pd.DataFrame(rows)
conflicts=images.groupby('patient_id').official.nunique(); conflicts=conflicts[conflicts>1].index.tolist()
image_conflicts=int(images.official_conflict.sum())
if (conflicts or image_conflicts) and not cfg.repartition_all_patients:
    pd.Series(conflicts,name='patient_id').to_csv(RUN/'patient_split_conflicts.csv',index=False)
    print(f'{len(conflicts)} patients and {image_conflicts} images cross the official train/test files (known for CBIS-DDSM). '
          'Falling back to a patient-level 70/15/15 split; official partitions are NOT used. Report this in the methods.')
    cfg.repartition_all_patients=True; json.dump(asdict(cfg),open(RUN/'config.json','w'),indent=2)
if cfg.repartition_all_patients:
    a,b=next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=cfg.seed).split(images,groups=images.patient_id))
    images['split']='test'; sub=images.iloc[a]
    c,d=next(GroupShuffleSplit(n_splits=1,test_size=.15/.85,random_state=cfg.seed+1).split(sub,groups=sub.patient_id))
    images.loc[sub.iloc[c].index,'split']='train'; images.loc[sub.iloc[d].index,'split']='val'
else:
    images['split']=images.official
    if not (images.split=='val').any():
        sub=images[images.split=='train']
        a,b=next(GroupShuffleSplit(n_splits=1,test_size=cfg.validation_fraction_of_train,random_state=cfg.seed).split(sub,groups=sub.patient_id))
        images.loc[sub.iloc[b].index,'split']='val'
if images.groupby('patient_id').split.nunique().max()!=1: raise ValueError('Patient leakage')
for split in ['train','val','test']:
    subset=images[images.split==split]
    if not len(subset): raise ValueError(f'Empty {split} split')
    if split!='test' and subset.label.dropna().nunique()!=2: raise ValueError(f'{split} requires both diagnostic classes')
images.to_csv(RUN/'image_manifest.csv',index=False); raw.to_csv(RUN/'resolved_abnormality_manifest.csv',index=False)
display(images.groupby('split').agg(images=('image_id','size'),patients=('patient_id','nunique'),abnormalities=('abnormalities','sum'),known_labels=('label','count'),mixed_images=('mixed','sum')))
display(images.groupby('split').label.value_counts(dropna=False))

## 3. Image/mask registration and preprocessing
ROI arrays must match full-image dimensions. Do not resize a cropped ROI to pretend it is a full-image mask. Binary-like ROI checks reject likely lesion photographs, but visual review is still necessary. Distinct overlapping lesion IDs require a richer annotation adapter and are rejected rather than overwritten.

Images and type/instance masks receive the same spatial resize. If downsampling removes a complete annotated lesion, the code stops: do not silently delete the lesion. Increase configured resolution or use registered native-resolution crops.

DICOM image normalization uses the configured percentile window and MONOCHROME1 inversion; confirm intensity and orientation visually. Non-DICOM uint8/uint16 intensity uses its dtype range. Caches are fingerprinted by file size/mtime, resolution and preprocessing settings.

In [ ]:
def read_array(path):
    path=Path(path)
    if path.suffix.lower() in ['.dcm','.dicom','']:
        try:
            ds=pydicom.dcmread(path); a=np.asarray(ds.pixel_array)
            if a.ndim!=2: raise ValueError(f'Expected 2D DICOM {path}')
            return a,ds
        except pydicom.errors.InvalidDicomError:
            if path.suffix.lower() in ['.dcm','.dicom']: raise
    im=Image.open(path)
    if im.mode in ['RGB','RGBA','P']: im=im.convert('L')
    a=np.asarray(im)
    if a.ndim!=2: raise ValueError(f'Expected grayscale image {path}')
    return a,None

def image_intensity(a,ds):
    if ds is not None:
        x=a.astype(np.float32)*float(getattr(ds,'RescaleSlope',1))+float(getattr(ds,'RescaleIntercept',0))
        lo,hi=np.percentile(x,cfg.dicom_percentiles); x=np.clip((x-lo)/max(hi-lo,1e-6),0,1)
        return 1-x if str(getattr(ds,'PhotometricInterpretation',''))=='MONOCHROME1' else x
    if a.dtype in [np.dtype('uint8'),np.dtype('uint16')]: return a.astype(np.float32)/np.iinfo(a.dtype).max
    x=a.astype(np.float32)
    if x.min()>=0 and x.max()<=1: return x
    raise ValueError(f'Unsupported image dtype/range {a.dtype}; supply explicit bit-depth normalization')

def _read_mask(path,shape):
    a,_=read_array(path)
    if a.shape!=shape: raise ValueError(f'ROI registration mismatch: {path}, ROI {a.shape}, full image {shape}')
    x=a.astype(np.float32)
    if not np.isfinite(x).all() or x.min()<0 or x.max()<=0: raise ValueError(f'Empty/invalid ROI {path}')
    x=x/x.max()
    if np.mean((x<.1)|(x>.9))<.98: raise ValueError(f'Not a binary-like mask: {path}; check roi_file versus crop_file')
    mask=x>cfg.mask_threshold_fraction
    if not mask.any() or mask.mean()>.8: raise ValueError(f'Unexpected mask foreground {path}')
    return mask

SWAPPED=[]
def read_roi(path,shape):
    try: return _read_mask(path,shape)
    except ValueError as first_error:
        alternative=ALT_ROI.get(str(path))
        if alternative:
            try: mask=_read_mask(alternative,shape); SWAPPED.append((str(path),alternative)); return mask
            except ValueError: pass
        raise first_error

prepared_rows=[]; duplicate_splits={}
for number,row in images.iterrows():
    group=raw[raw.full_file_resolved==row.full_file]; folder=CACHE/row.image_id; folder.mkdir(exist_ok=True)
    image_path=folder/'image.png'; ann_path=folder/'annotation.npz'; meta_path=folder/'metadata.json'
    paths=[row.full_file]+sorted(group.roi_file_resolved.unique())
    signature=dict(files=[(str(p),Path(p).stat().st_size) for p in paths],
        lesions=[(str(r.abnormality_id),int(r.type_code),r.roi_file_resolved) for r in group.itertuples()],
        size=[cfg.height,cfg.width],window=cfg.dicom_percentiles,mask_threshold=cfg.mask_threshold_fraction)
    fingerprint=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    valid=not cfg.rebuild_cache and image_path.exists() and ann_path.exists() and meta_path.exists()
    if valid: valid=json.load(open(meta_path)).get('fingerprint')==fingerprint
    if not valid:
        a,ds=read_array(row.full_file); shape=a.shape; x=image_intensity(a,ds)
        target=np.zeros((2,*shape),dtype=np.uint8); instances=np.zeros(shape,dtype=np.int32); lesion_metadata=[]; instance_id=1
        for (type_code,abnormality_id),records in group.groupby(['type_code','abnormality_id'],sort=False):
            region=np.zeros(shape,dtype=bool)
            for roi_path in records.roi_file_resolved.unique(): region|=read_roi(roi_path,shape)
            if np.any(region&(instances>0)): raise ValueError(f'Overlapping distinct lesion annotations require an explicit adapter: {row.full_file}')
            target[int(type_code)-1,region]=1; instances[region]=instance_id
            lesion_metadata.append(dict(instance=instance_id,type_code=int(type_code),native_area=int(region.sum()),abnormality_id=str(abnormality_id)))
            instance_id+=1
        size=(cfg.width,cfg.height)
        Image.fromarray((x*255).round().astype(np.uint8)).resize(size,Image.Resampling.BILINEAR).save(image_path)
        target=np.stack([np.asarray(Image.fromarray(t).resize(size,Image.Resampling.NEAREST)) for t in target]).astype(np.uint8)
        instances=np.asarray(Image.fromarray(instances).resize(size,Image.Resampling.NEAREST)).astype(np.int32)
        if set(range(1,instance_id))!=(set(np.unique(instances))-{0}): raise ValueError(f'Downsampling erased lesions: {row.full_file}; increase resolution')
        if not np.array_equal(instances>0,target.any(0)): raise ValueError('Mask alignment mismatch after resizing')
        np.savez_compressed(ann_path,target=target,instances=instances)
        json.dump(dict(fingerprint=fingerprint,native_shape=list(shape),lesions=lesion_metadata),open(meta_path,'w'),indent=2)
    digest=hashlib.sha256(image_path.read_bytes()).hexdigest()
    if digest in duplicate_splits and duplicate_splits[digest]!=row.split: raise ValueError('Exact processed duplicate crosses splits')
    duplicate_splits[digest]=row.split
    prepared_rows.append(dict(**row.to_dict(),image_path=str(image_path),annotation_path=str(ann_path),metadata_path=str(meta_path)))
    if (number+1)%100==0: print('Prepared',number+1,'/',len(images))
prepared=pd.DataFrame(prepared_rows); prepared.to_csv(RUN/'prepared_manifest.csv',index=False)
print('ROI files replaced by their crop file (ROI/crop mix-up) in this run:',len(SWAPPED)); pd.DataFrame(SWAPPED,columns=['roi_as_listed','used_instead']).to_csv(RUN/'roi_crop_swaps.csv',index=False)

class MammogramDataset(Dataset):
    def __init__(self,table,training=False): self.table=table.reset_index(drop=True); self.training=training
    def __len__(self): return len(self.table)
    def __getitem__(self,index):
        r=self.table.iloc[index]; x=np.asarray(Image.open(r.image_path),dtype=np.float32)/255
        with np.load(r.annotation_path) as z: target=z['target'].astype(np.float32); instances=z['instances'].astype(np.int64)
        if self.training:
            if random.random()<.5: x=x[:,::-1].copy(); target=target[:,:,::-1].copy(); instances=instances[:,::-1].copy()
            x=np.clip(x*random.uniform(.95,1.05),0,1)
        return dict(x=torch.from_numpy(x.copy())[None],target=torch.from_numpy(target.copy()),instances=torch.from_numpy(instances.copy()),
                    label=torch.tensor(float(r.label),dtype=torch.float32),patient=str(r.patient_id),image=str(r.image_id))

def worker_seed(worker_id):
    s=torch.initial_seed()%2**32; random.seed(s); np.random.seed(s)
loaders={s:DataLoader(MammogramDataset(prepared[prepared.split==s],s=='train'),batch_size=cfg.batch_size,shuffle=s=='train',
    num_workers=cfg.workers,worker_init_fn=worker_seed,generator=torch.Generator().manual_seed(cfg.seed),pin_memory=DEVICE.type=='cuda') for s in ['train','val','test']}

fig,axes=plt.subplots(min(4,len(loaders['val'].dataset)),3,figsize=(12,14),squeeze=False)
for i in range(len(axes)):
    sample=loaders['val'].dataset[i]
    for j in range(3): axes[i,j].imshow(sample['x'][0],cmap='gray'); axes[i,j].axis('off')
    axes[i,0].set_title('Image '+sample['image'])
    for t in range(2):
        if sample['target'][t].any(): axes[i,t+1].contour(sample['target'][t],levels=[.5],colors='red')
        axes[i,t+1].set_title(['Mass annotation','Calcification annotation'][t])
plt.tight_layout(); fig.savefig(RUN/'registration_audit.png',dpi=140); plt.show()
# Inspect the overlays. Stop execution if masks/intensity are incorrect.

## 4. Localization model and objective
Architecture: image + local high-pass image → full-resolution detail stream → half-scale adaptive dilation branches (1/2/4) + quarter-scale context → full-resolution fusion → two sigmoid lesion maps. No received-region input exists.

For each image/type, W gives each foreground instance pixel weight 1/sqrt(instance area), background weight 1/sqrt(background area). The objective is the sum of squared Gaussian-smoothed weighted residuals at scales 0,1,3, divided by instance count plus one. It is an adapted quadratic objective, not a novel loss claim. Sigmoid saturation and insufficient image resolution remain limitations.

In [ ]:
def conv(cin,cout,stride=1,dilation=1):
    return nn.Sequential(nn.Conv2d(cin,cout,3,stride,padding=dilation,dilation=dilation,bias=False),nn.GroupNorm(4,cout),nn.GELU())

class LesionFields(nn.Module):
    def __init__(self):
        super().__init__(); self.detail=conv(2,16); self.down=conv(16,24,stride=2)
        self.context=nn.Sequential(conv(24,32,stride=2),conv(32,32,dilation=2)); self.project=nn.Conv2d(32,24,1)
        self.branches=nn.ModuleList([nn.Sequential(nn.Conv2d(24,24,3,padding=d,dilation=d,groups=24),nn.Conv2d(24,24,1),nn.GELU()) for d in [1,2,4]])
        self.gate=nn.Conv2d(24,3,1); self.head=nn.Sequential(conv(40,24),nn.Conv2d(24,2,1))
    def forward(self,x):
        full=self.detail(torch.cat([x,x-F.avg_pool2d(x,5,1,2)],1)); half=self.down(full); weights=self.gate(half).softmax(1)
        mixed=sum(weights[:,i:i+1]*branch(half) for i,branch in enumerate(self.branches))
        mixed=mixed+F.interpolate(self.project(self.context(half)),size=half.shape[-2:],mode='bilinear',align_corners=False)
        logits=self.head(torch.cat([full,F.interpolate(mixed,size=x.shape[-2:],mode='bilinear',align_corners=False)],1))
        p=logits.sigmoid(); return dict(p=p,union=1-(1-p).prod(1,keepdim=True),logits=logits)

def gaussian(x,sigma):
    if sigma==0: return x
    radius=math.ceil(3*sigma); z=torch.arange(-radius,radius+1,device=x.device,dtype=x.dtype)
    k=torch.exp(-.5*(z/sigma)**2); k=k/k.sum()
    x=F.conv2d(x,k.reshape(1,1,1,-1),padding=(0,radius))
    return F.conv2d(x,k.reshape(1,1,-1,1),padding=(radius,0))

def localization_loss(p,target,instances):
    terms=[]
    for b in range(len(p)):
        for t in range(2):
            y=target[b:b+1,t:t+1]; ids=torch.unique(instances[b][y[0,0]>0]); ids=ids[ids>0]
            if cfg.balance_instances:
                weight=torch.where(y==0,(y==0).sum().clamp_min(1).float().rsqrt(),torch.zeros_like(y))
                for j in ids:
                    region=(instances[b]==j)[None,None]&(y>0)
                    weight=torch.where(region,region.sum().clamp_min(1).float().rsqrt(),weight)
                denominator=len(ids)+1
            else: weight=torch.ones_like(y)/math.sqrt(y.numel()); denominator=1
            residual=weight*(p[b:b+1,t:t+1]-y)
            terms.append(sum(gaussian(residual,s).square().sum() for s in cfg.localization_scales)/denominator)
    return torch.stack(terms).mean()

## 5. Image classification model and objective
A shallow appearance extractor produces region tokens on a 24×16 grid. Tokens receive image position and predicted lesion statistics (two type means and maximum lesion probability). Four latent queries integrate local evidence; a diagnostic query plus global token produces a malignancy probability.

These latent tokens are not validated clinical concepts. Localization probabilities are predicted from the input image, never oracle masks. Localization is frozen during classifier training.

Classification objective: average Brier error within each present diagnosis class, then average those class means. Missing diagnoses are excluded. This changes the effective training prior; validation calibration must be measured. No simulated corruption or scenario-tail objective is used.

In [ ]:
class ImageTokenClassifier(nn.Module):
    def __init__(self):
        super().__init__(); self.features=nn.Sequential(conv(2,16,stride=2),conv(16,32,stride=2),conv(32,64,stride=2))
        self.embed=nn.Linear(67,64); self.position=nn.Linear(2,64)
        self.latent=nn.Parameter(torch.randn(1,4,64)*.02); self.query=nn.Parameter(torch.randn(1,1,64)*.02)
        self.attention=nn.MultiheadAttention(64,4,batch_first=True); self.diagnostic=nn.MultiheadAttention(64,4,batch_first=True)
        self.norm=nn.LayerNorm(64); self.head=nn.Sequential(nn.Linear(128,64),nn.GELU(),nn.Linear(64,1))
    def forward(self,x,localization):
        B=x.shape[0]; h,w=24,16
        appearance=F.adaptive_avg_pool2d(self.features(torch.cat([x,x-F.avg_pool2d(x,5,1,2)],1)),(h,w))
        stats=torch.cat([F.adaptive_avg_pool2d(localization,(h,w)),F.adaptive_max_pool2d(localization.max(1,keepdim=True).values,(h,w))],1)
        z=self.embed(torch.cat([appearance,stats],1).flatten(2).transpose(1,2))
        yy,xx=torch.meshgrid(torch.linspace(-1,1,h,device=x.device),torch.linspace(-1,1,w,device=x.device),indexing='ij')
        z=z+self.position(torch.stack([yy,xx],-1).reshape(1,h*w,2)); global_z=z.mean(1,keepdim=True); keys=torch.cat([z,global_z],1)
        prior=stats[:,:2].max(1).values.flatten(1); bias=(.5*torch.log(.1+prior)).clamp(-2,0)
        bias=torch.cat([bias,torch.zeros(B,1,device=x.device)],1)
        mask=bias[:,None,None,:].expand(B,4,4,h*w+1).reshape(B*4,4,h*w+1)
        q=self.latent.expand(B,-1,-1); latent,_=self.attention(q,keys,keys,attn_mask=mask,need_weights=False)
        latent=self.norm(latent+q); all_keys=torch.cat([keys,latent],1)
        d,_=self.diagnostic(self.query.expand(B,-1,-1),all_keys,all_keys,need_weights=False)
        logit=self.head(torch.cat([d[:,0],global_z[:,0]],-1)).squeeze(-1)
        return dict(prob=logit.sigmoid(),logit=logit)

def classification_loss(prob,label):
    terms=[]
    for c in [0,1]:
        known=torch.isfinite(label)&(label==c)
        if known.any(): terms.append((prob[known]-c).square().mean())
    return torch.stack(terms).mean() if terms else prob.sum()*0

## 6. Initialization, optimizers, checkpointing and smoke tests
Models train from scratch using PyTorch defaults for convolution/linear/normalization/attention layers. Latent queries explicitly use Normal(0, 0.02²). No ImageNet weights are loaded. SGD momentum is default; AdamW and Lion are available with separately tuned learning rates.

Epoch-boundary resume saves optimizer, scheduler, RNG and DataLoader state. A fingerprint rejects changes to data, split, or training configuration; use a new run folder when changing an experiment. Exact reproducibility across GPU/library versions is not guaranteed.

In [ ]:
class Lion(torch.optim.Optimizer):
    def __init__(self,params,lr=1e-4,weight_decay=0): super().__init__(params,dict(lr=lr,weight_decay=weight_decay))
    @torch.no_grad()
    def step(self,closure=None):
        loss=None
        if closure is not None:
            with torch.enable_grad(): loss=closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                if p.grad.is_sparse: raise ValueError('Sparse gradients unsupported')
                state=self.state[p]
                if 'm' not in state: state['m']=torch.zeros_like(p)
                m=state['m']; p.mul_(1-group['lr']*group['weight_decay'])
                p.add_((.9*m+.1*p.grad).sign(),alpha=-group['lr']); m.mul_(.99).add_(p.grad,alpha=.01)
        return loss

def optimizer_for(params):
    if cfg.optimizer=='sgd': return torch.optim.SGD(params,lr=cfg.learning_rate,momentum=.9,weight_decay=cfg.weight_decay)
    if cfg.optimizer=='adamw': return torch.optim.AdamW(params,lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    if cfg.optimizer=='lion': return Lion(params,lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    raise ValueError(cfg.optimizer)
def device_batch(batch): return {k:v.to(DEVICE) if torch.is_tensor(v) else v for k,v in batch.items()}
locnet=LesionFields().to(DEVICE); classifier=ImageTokenClassifier().to(DEVICE)
config_for_hash=asdict(cfg); config_for_hash.pop('resume'); config_for_hash.pop('rebuild_cache')
fingerprints=[json.load(open(p))['fingerprint'] for p in prepared.metadata_path]
experiment_hash=hashlib.sha256((prepared[['image_id','patient_id','split','label']].to_csv(index=False)+json.dumps(config_for_hash,sort_keys=True)+json.dumps(fingerprints)).encode()).hexdigest()

def save_checkpoint(path,model,opt,schedule,epoch,best,stale,history):
    state=dict(model=model.state_dict(),optimizer=opt.state_dict(),scheduler=schedule.state_dict(),epoch=epoch,
        best=best,stale=stale,history=history,experiment_hash=experiment_hash,
        torch_rng=torch.get_rng_state(),numpy_rng=np.random.get_state(),python_rng=random.getstate(),
        cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],loader_rng={k:v.generator.get_state() for k,v in loaders.items()})
    temp=Path(str(path)+'.tmp'); torch.save(state,temp); temp.replace(path)

def load_checkpoint(path,model,opt=None,schedule=None,restore_rng=False):
    # Trust only checkpoints you created; pickle metadata is included.
    state=torch.load(path,map_location=DEVICE,weights_only=False)
    if state['experiment_hash']!=experiment_hash: raise ValueError('Checkpoint data/config mismatch; create a new run directory')
    model.load_state_dict(state['model'])
    if opt is not None: opt.load_state_dict(state['optimizer'])
    if schedule is not None: schedule.load_state_dict(state['scheduler'])
    if restore_rng:
        torch.set_rng_state(state['torch_rng'].cpu()); np.random.set_state(state['numpy_rng']); random.setstate(state['python_rng'])
        if torch.cuda.is_available() and state['cuda_rng']: torch.cuda.set_rng_state_all([x.cpu() for x in state['cuda_rng']])
        for k,v in state['loader_rng'].items(): loaders[k].generator.set_state(v.cpu())
    return state

def model_signature(model):
    h=hashlib.sha256()
    for value in model.state_dict().values(): h.update(value.detach().cpu().numpy().tobytes())
    return h.hexdigest()

def smoke_test():
    l=LesionFields().to(DEVICE); c=ImageTokenClassifier().to(DEVICE); x=torch.rand(2,1,64,64,device=DEVICE)
    p=l(x)['p']; out=c(x,p); assert p.shape==(2,2,64,64) and out['prob'].shape==(2,)
    y=torch.zeros_like(p); y[:,0,20:23,20:23]=1; inst=torch.zeros(2,64,64,dtype=torch.long,device=DEVICE); inst[:,20:23,20:23]=1
    loss=localization_loss(p,y,inst)+classification_loss(out['prob'],torch.tensor([0.,1.],device=DEVICE))
    loss.backward(); assert torch.isfinite(loss)
    assert classification_loss(out['prob'],torch.full((2,),float('nan'),device=DEVICE)).item()==0
    return dict(localizer_parameters=sum(p.numel() for p in l.parameters()),classifier_parameters=sum(p.numel() for p in c.parameters()))
display(smoke_test()); seed_all(cfg.seed)

## 7. Localization training and classifier training
Localization checkpoint selection: lowest validation localization loss. Classification checkpoint selection: highest validation ROC-AUC. Early stopping and cosine schedules use validation only.

The classifier always uses predicted maps from the frozen selected localizer. Ground-truth masks are not classifier inputs. Binary-ambiguous images are excluded from the classifier training loader but remain in localization training.

In [ ]:
@torch.no_grad()
def validation_localization():
    locnet.eval(); total=0.; count=0
    for batch in loaders['val']:
        b=device_batch(batch); loss=localization_loss(locnet(b['x'])['p'],b['target'],b['instances'])
        total+=loss.item()*len(b['x']); count+=len(b['x'])
    return total/max(count,1)

@torch.no_grad()
def predict(loader):
    locnet.eval(); classifier.eval(); rows=[]
    for batch in loader:
        b=device_batch(batch); out=classifier(b['x'],locnet(b['x'])['p'])['prob'].cpu().numpy()
        for i,p in enumerate(out): rows.append(dict(patient=b['patient'][i],image=b['image'][i],label=float(b['label'][i]),prob=float(p)))
    return pd.DataFrame(rows)

cls_training=prepared[(prepared.split=='train')&prepared.label.notna()]
cls_loader=DataLoader(MammogramDataset(cls_training,True),batch_size=cfg.batch_size,shuffle=True,num_workers=cfg.workers,
    worker_init_fn=worker_seed,generator=torch.Generator().manual_seed(cfg.seed+1),pin_memory=DEVICE.type=='cuda')
# Include the classifier sampler generator in checkpoint state.
loaders['classification_train']=cls_loader

def fit(stage,epochs):
    model=locnet if stage=='localization' else classifier
    if stage=='classification':
        signature=model_signature(locnet); signature_path=RUN/'classifier_localizer_signature.json'
        if cfg.resume and (RUN/'classification_last.pt').exists():
            if json.load(open(signature_path))['signature']!=signature: raise ValueError('Classifier checkpoint uses a different localizer')
        json.dump(dict(signature=signature),open(signature_path,'w'))
    opt=optimizer_for(model.parameters()); schedule=torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs)
    last=RUN/f'{stage}_last.pt'; bestfile=RUN/f'{stage}_best.pt'; start=0; stale=0; history=[]
    best=float('inf') if stage=='localization' else -float('inf')
    if cfg.resume and last.exists():
        state=load_checkpoint(last,model,opt,schedule,True)
        start=state['epoch']+1; best=state['best']; stale=state['stale']; history=state['history']; print('Resuming',stage,start+1)
    if stale>=cfg.patience:
        load_checkpoint(bestfile,model); print(stage,'already early-stopped'); return
    for epoch in range(start,epochs):
        model.train(); losses=[]; began=time.time()
        if stage=='classification': locnet.eval()
        loader=loaders['train'] if stage=='localization' else cls_loader
        for step,batch in enumerate(loader):
            b=device_batch(batch); opt.zero_grad(set_to_none=True)
            if stage=='localization': loss=localization_loss(locnet(b['x'])['p'],b['target'],b['instances'])
            else:
                with torch.no_grad(): maps=locnet(b['x'])['p']
                loss=classification_loss(classifier(b['x'],maps)['prob'],b['label'])
            if not torch.isfinite(loss): raise FloatingPointError(f'Nonfinite {stage} loss')
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5); opt.step(); losses.append(loss.item())
            if (step+1)%100==0: print(stage,'epoch',epoch+1,'batch',step+1,'loss',float(np.mean(losses[-100:])))
        if stage=='localization': score=validation_localization(); improved=score<best
        else:
            val=predict(loaders['val']).dropna(subset=['label']); score=roc_auc_score(val.label,val.prob); improved=score>best
        schedule.step(); stale=0 if improved else stale+1
        if improved: best=score
        history.append(dict(epoch=epoch+1,train_loss=float(np.mean(losses)),validation_score=float(score),seconds=time.time()-began))
        if improved: save_checkpoint(bestfile,model,opt,schedule,epoch,best,stale,history)
        save_checkpoint(last,model,opt,schedule,epoch,best,stale,history)
        pd.DataFrame(history).to_csv(RUN/f'{stage}_history.csv',index=False); print(stage,history[-1])
        if stale>=cfg.patience: print('Early stopping'); break
    if not bestfile.exists(): raise RuntimeError('No checkpoint created')
    load_checkpoint(bestfile,model)

if RUN_TRAINING:
    fit('localization',cfg.localization_epochs)
    fit('classification',cfg.classification_epochs)

## 8. Metrics and validation-only thresholds
Classification: ROC-AUC, average precision (reported as PR-AUC), sensitivity, specificity, F1, balanced accuracy, Brier, 10-bin ECE, confusion counts. Classification threshold maximizes validation balanced accuracy.

Localization: Dice/IoU, lesion sensitivity, FP/image, hit rate, detected-lesion center error and small-lesion sensitivity. Localization threshold maximizes validation mean union Dice. One-to-one matching maximizes admissible match count first, then IoU. Default matching IoU is 0.1, a declared experimental choice, not a clinical standard.

Small-lesion cutoff is currently resized mask area <=100 pixels. Native metadata is saved; replace this with a prespecified native/physical-size definition for a publication. Calcification predicted components may fragment one clinical cluster; report that limitation rather than treating each deposit as a clinical lesion.

In [ ]:
def classification_metrics(data,threshold=.5):
    q=data.dropna(subset=['label','prob']); y=q.label.to_numpy().astype(int); p=q.prob.to_numpy()
    if not len(y): return {}
    pred=(p>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); ece=0.
    for j in range(10):
        mask=(p>=j/10)&((p<(j+1)/10) if j<9 else (p<=1))
        if mask.any(): ece+=mask.mean()*abs(p[mask].mean()-y[mask].mean())
    return dict(roc_auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
        pr_auc=average_precision_score(y,p) if y.sum() else np.nan,sensitivity=tp/(tp+fn) if tp+fn else np.nan,
        specificity=tn/(tn+fp) if tn+fp else np.nan,f1=f1_score(y,pred,zero_division=0),
        balanced_accuracy=balanced_accuracy_score(y,pred),brier=float(np.mean((p-y)**2)),ece=float(ece),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))

@torch.no_grad()
def choose_localization_threshold():
    locnet.eval(); thresholds=np.linspace(.05,.95,19); totals=np.zeros(len(thresholds)); count=0
    for batch in loaders['val']:
        b=device_batch(batch); p=locnet(b['x'])['union']; y=b['target'].sum(1,keepdim=True).clamp_max(1)
        for j,t in enumerate(thresholds):
            mask=p>=t; totals[j]+=(2*(mask*y).sum((1,2,3))/(mask.sum((1,2,3))+y.sum((1,2,3))).clamp_min(1)).sum().item()
        count+=len(p)
    return float(thresholds[np.argmax(totals/max(count,1))])

def per_image_localization(prob,target,instances,threshold):
    pred=prob>=threshold; gt=target>0; components,n=ndimage.label(pred); ids=np.unique(instances); ids=ids[ids>0]
    overlap=np.zeros((len(ids),n)); distance=np.zeros_like(overlap); areas=[]
    for a,j in enumerate(ids):
        mask=instances==j; areas.append(mask.sum()); center=np.argwhere(mask).mean(0)
        for b in range(n):
            other=components==b+1; overlap[a,b]=np.logical_and(mask,other).sum()/max(np.logical_or(mask,other).sum(),1)
            distance[a,b]=np.linalg.norm(center-np.argwhere(other).mean(0))
    found=np.zeros(len(ids),dtype=bool); errors=[]
    if len(ids) and n:
        admissible=overlap>=cfg.matching_iou
        utility=admissible*(min(len(ids),n)+1)+overlap
        a,b=linear_sum_assignment(-utility); keep=admissible[a,b]; found[a[keep]]=True; errors=distance[a[keep],b[keep]].tolist()
    intersection=np.logical_and(pred,gt).sum(); union=np.logical_or(pred,gt).sum(); small=np.asarray(areas)<=cfg.small_lesion_resized_area
    return dict(dice=2*intersection/max(pred.sum()+gt.sum(),1),iou=intersection/max(union,1),lesions=len(ids),detected=int(found.sum()),
        false_positives=n-int(found.sum()),hits=sum(bool(pred[instances==j].any()) for j in ids),
        small_lesions=int(small.sum()),small_detected=int(found[small].sum()),center_error_sum=float(sum(errors)),center_error_count=len(errors))

@torch.no_grad()
def localization_evaluation(loader,threshold):
    locnet.eval(); rows=[]
    for batch in loader:
        b=device_batch(batch); out=locnet(b['x']); maps=out['p'].cpu().numpy(); union=out['union'].cpu().numpy()[:,0]
        target=b['target'].cpu().numpy(); instances=b['instances'].cpu().numpy()
        for i in range(len(union)):
            for category,t in [('union',None),('mass',0),('calcification',1)]:
                prob=union[i] if t is None else maps[i,t]; gt=target[i].any(0) if t is None else target[i,t]>0
                inst=instances[i].copy(); inst[~gt]=0
                row=per_image_localization(prob,gt,inst,threshold)
                # Empty/empty masks have Dice and IoU 0 here. Report lesion-present overlap separately.
                row.update(patient=b['patient'][i],image=b['image'][i],category=category,lesion_present=bool(gt.any()))
                rows.append(row)
    return pd.DataFrame(rows)

def summarize_localization(table):
    records=[]
    for category,q in table.groupby('category'):
        positive=q[q.lesion_present]; n=q.lesions.sum(); small=q.small_lesions.sum(); matched=q.center_error_count.sum()
        records.append(dict(category=category,images=len(q),lesion_present_images=len(positive),mean_dice_lesion_present=positive.dice.mean(),
            mean_iou_lesion_present=positive.iou.mean(),lesion_sensitivity=q.detected.sum()/n if n else np.nan,
            false_positives_per_image=q.false_positives.mean(),hit_rate=q.hits.sum()/n if n else np.nan,
            small_lesion_sensitivity=q.small_detected.sum()/small if small else np.nan,
            detected_lesion_center_error=q.center_error_sum.sum()/matched if matched else np.nan))
    return pd.DataFrame(records)

if RUN_TRAINING:
    validation=predict(loaders['val']); threshold=float(max(np.linspace(.01,.99,99),key=lambda t:classification_metrics(validation,t)['balanced_accuracy']))
    thresholds=dict(classification=threshold,localization=choose_localization_threshold())
    json.dump(thresholds,open(RUN/'thresholds.json','w'),indent=2); validation.to_csv(RUN/'validation_predictions.csv',index=False)
    display(pd.DataFrame([classification_metrics(validation,threshold)]))
    val_loc=localization_evaluation(loaders['val'],thresholds['localization']); val_loc.to_csv(RUN/'validation_localization.csv',index=False)
    display(summarize_localization(val_loc))

## 9. Confidence intervals, plots and final test gate
Confidence intervals resample patients with all their images, not individual images independently. AUC skips single-class bootstrap samples. These intervals do not include training-seed variability; retrain independently for that analysis.

Test evaluation is deliberately opt-in. Do not select architecture, thresholds, learning rates or checkpoints using test performance. In a fresh runtime, rerun preprocessing/model cells with `RUN_TRAINING=False`, load checkpoints below and enable `RUN_TEST=True` only after settings are frozen.

In [ ]:
def classification_bootstrap(data,threshold,repetitions=1000):
    rng=np.random.default_rng(cfg.seed); patients=data.patient.unique(); values=[]
    for _ in range(repetitions):
        sample=pd.concat([data[data.patient==p] for p in rng.choice(patients,len(patients),replace=True)],ignore_index=True)
        values.append(classification_metrics(sample,threshold))
    table=pd.DataFrame(values); records=[]
    for name in ['roc_auc','pr_auc','sensitivity','specificity','f1','balanced_accuracy','brier','ece']:
        v=table[name].dropna()
        records.append(dict(metric=name,lower=float(v.quantile(.025)) if len(v) else np.nan,upper=float(v.quantile(.975)) if len(v) else np.nan))
    return pd.DataFrame(records)

def localization_bootstrap(table,repetitions=1000):
    rng=np.random.default_rng(cfg.seed); records=[]
    for category,q in table.groupby('category'):
        patients=q.patient.unique(); samples=[]
        for _ in range(repetitions):
            selected=pd.concat([q[q.patient==p] for p in rng.choice(patients,len(patients),replace=True)],ignore_index=True)
            samples.append(summarize_localization(selected).iloc[0].to_dict())
        samples=pd.DataFrame(samples)
        for metric in ['mean_dice_lesion_present','mean_iou_lesion_present','lesion_sensitivity','false_positives_per_image','hit_rate','small_lesion_sensitivity','detected_lesion_center_error']:
            v=pd.to_numeric(samples[metric],errors='coerce').dropna()
            records.append(dict(category=category,metric=metric,lower=float(v.quantile(.025)) if len(v) else np.nan,upper=float(v.quantile(.975)) if len(v) else np.nan))
    return pd.DataFrame(records)

def classification_plots(data,threshold,prefix):
    q=data.dropna(subset=['label']); y=q.label.to_numpy(); p=q.prob.to_numpy(); fig,axes=plt.subplots(1,3,figsize=(14,4))
    if len(np.unique(y))==2:
        fpr,tpr,_=roc_curve(y,p); axes[0].plot(fpr,tpr); axes[0].plot([0,1],[0,1],'--',color='gray')
        precision,recall,_=precision_recall_curve(y,p); axes[1].plot(recall,precision)
    axes[0].set(xlabel='False-positive rate',ylabel='Sensitivity',title='ROC'); axes[1].set(xlabel='Recall',ylabel='Precision',title='Precision-recall')
    means=[]; observed=[]
    for j in range(10):
        mask=(p>=j/10)&((p<(j+1)/10) if j<9 else (p<=1))
        if mask.any(): means.append(p[mask].mean()); observed.append(y[mask].mean())
    axes[2].plot(means,observed,'o-'); axes[2].plot([0,1],[0,1],'--',color='gray'); axes[2].set(xlabel='Mean probability',ylabel='Observed malignancy',title='Reliability')
    fig.tight_layout(); fig.savefig(RUN/f'{prefix}_classification_curves.png',dpi=160); plt.show()
    matrix=confusion_matrix(y,p>=threshold,labels=[0,1]); fig,ax=plt.subplots(figsize=(4,4)); ax.imshow(matrix,cmap='Blues')
    for i in range(2):
        for j in range(2): ax.text(j,i,str(matrix[i,j]),ha='center',va='center')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Benign','Malignant'],yticklabels=['Benign','Malignant'],xlabel='Predicted',ylabel='True')
    fig.tight_layout(); fig.savefig(RUN/f'{prefix}_confusion_matrix.png',dpi=150); plt.show()

@torch.no_grad()
def localization_plots(loader,threshold,prefix,n=4):
    locnet.eval(); ds=loader.dataset; n=min(n,len(ds)); fig,axes=plt.subplots(n,3,figsize=(11,4*n),squeeze=False)
    for i in range(n):
        b=ds[i]; p=locnet(b['x'][None].to(DEVICE))['union'][0,0].cpu().numpy()
        axes[i,0].imshow(b['x'][0],cmap='gray'); axes[i,1].imshow(b['target'].sum(0)>0,cmap='gray'); axes[i,2].imshow(p,vmin=0,vmax=1,cmap='magma')
        if (p>=threshold).any(): axes[i,2].contour(p>=threshold,levels=[.5],colors='cyan')
        for j,title in enumerate(['Image','Ground truth','Predicted lesion probability']): axes[i,j].set_title(title); axes[i,j].axis('off')
    fig.tight_layout(); fig.savefig(RUN/f'{prefix}_localization_examples.png',dpi=150); plt.show()

if RUN_TRAINING:
    classification_plots(validation,thresholds['classification'],'validation')
    localization_plots(loaders['val'],thresholds['localization'],'validation')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for stage,ax in zip(['localization','classification'],axes):
        history=pd.read_csv(RUN/f'{stage}_history.csv'); ax.plot(history.epoch,history.train_loss,label='Training loss'); ax.set_title(stage); ax.set_xlabel('Epoch'); ax.legend()
    fig.tight_layout(); fig.savefig(RUN/'training_losses.png',dpi=150); plt.show()

if RUN_TEST:
    load_checkpoint(RUN/'localization_best.pt',locnet); load_checkpoint(RUN/'classification_best.pt',classifier)
    if json.load(open(RUN/'classifier_localizer_signature.json'))['signature']!=model_signature(locnet): raise ValueError('Classifier/localizer checkpoint mismatch')
    thresholds=json.load(open(RUN/'thresholds.json'))
    test=predict(loaders['test']); test.to_csv(RUN/'test_classification_predictions.csv',index=False)
    cls_summary=pd.DataFrame([classification_metrics(test,thresholds['classification'])]); cls_summary.to_csv(RUN/'test_classification_summary.csv',index=False)
    classification_bootstrap(test,thresholds['classification']).to_csv(RUN/'test_classification_ci95.csv',index=False)
    loc_table=localization_evaluation(loaders['test'],thresholds['localization']); loc_table.to_csv(RUN/'test_localization_per_image.csv',index=False)
    loc_summary=summarize_localization(loc_table); loc_summary.to_csv(RUN/'test_localization_summary.csv',index=False)
    localization_bootstrap(loc_table).to_csv(RUN/'test_localization_ci95.csv',index=False)
    display(cls_summary); display(loc_summary)
    classification_plots(test,thresholds['classification'],'test'); localization_plots(loaders['test'],thresholds['localization'],'test')

## 10. Experiment controls and limitations
- Use a new `RUN` directory for every seed, optimizer, loss or architecture ablation; train independently and tune only on validation data.
- Supported loss ablations: `localization_scales=(0,)` versus `(0,1,3)`; `balance_instances=False/True`.
- External baselines (your previous CLEAN_V2/Pheno, U-Net/ResNet, transformer segmentation, GMIC) are not implemented in this notebook. Do not call this an automatic comparison against those methods.
- No full FROC sweep is implemented. FP/image and sensitivity at the frozen threshold are available.
- Descriptor labels are absent; learned queries cannot be interpreted as supervised mass-shape or calcification-type concepts.
- CBIS-DDSM is abnormality enriched; do not equate image specificity here to population screening specificity.
- Masks may denote calcification-cluster ROIs, not individual calcifications. Registered resolution reduction may erase fine diagnostic structure even if an ROI survives.
- Calibration is evaluated but no calibration map or validated uncertainty estimator is fitted.
- Read all printed checks and inspect mask overlays. A running program is not evidence of valid clinical annotation mapping.
- Training is enabled, but generation-time tensor execution was not performed. Run smoke tests locally and investigate any exception rather than bypassing data-integrity checks.

All outputs are stored in your configured `mammo_semcom_work/runs/imaging_only_v1` folder; processed images/masks are in its dedicated imaging cache.